In [4]:
from pathlib import Path
from collections import Counter
import hashlib
import json
import shutil
import sqlite3
import subprocess
import zipfile

import torch
from google.colab import drive
from tqdm.auto import tqdm

drive.mount("/content/drive")
assert torch.cuda.is_available(), "Select a GPU runtime first."

drive_root = Path("/content/drive/MyDrive/FacialVisualProfile")
workspace = Path("/content/m3_seed0_workspace")
data_dir = Path("/content/m3_seed0_data")
raw_root = data_dir / "raw"
image_dir = raw_root / "img_align_celeba"
expected_commit = "44e8ac2"

print("GPU:", torch.cuda.get_device_name(0))
print("PyTorch:", torch.__version__)

if not workspace.exists():
    subprocess.run([
        "git", "clone", "--no-checkout",
        "https://github.com/DeweyYip/facial-visual-profile.git",
        str(workspace),
    ], check=True)
    subprocess.run([
        "git", "-C", str(workspace),
        "checkout", "--detach", expected_commit,
    ], check=True)

status = subprocess.check_output(
    ["git", "-C", str(workspace), "status", "--porcelain"],
    text=True,
)
assert not status.strip(), "Workspace has local changes."

commit = subprocess.check_output(
    ["git", "-C", str(workspace), "rev-parse", "HEAD"],
    text=True,
).strip()
assert commit.startswith(expected_commit), commit
print("Source commit:", commit)

# Recheck the sources that passed baseline compatibility checks.
compatibility = json.loads(
    (drive_root / "runs/m3_preparation/day11/source_compatibility.json")
    .read_text(encoding="utf-8")
)
for record in compatibility["files"]:
    path = workspace / record["file"]
    assert hashlib.sha256(path.read_bytes()).hexdigest() == (
        record["current_sha256"]
    ), record["file"]

# Recheck model and Dataset sources against the successful overfit run.
overfit = json.loads(
    (drive_root / "runs/m3_preparation/day11/overfit_report.json")
    .read_text(encoding="utf-8")
)
assert overfit["result"] == "PASS"
for name, expected_hash in overfit["source_sha256"].items():
    assert hashlib.sha256(
        (workspace / name).read_bytes()
    ).hexdigest() == expected_hash, name

image_dir.mkdir(parents=True, exist_ok=True)
for name in [
    "list_attr_celeba.txt",
    "list_landmarks_align_celeba.txt",
    "list_eval_partition.txt",
]:
    shutil.copy2(drive_root / "data" / name, raw_root / name)

cache_path = data_dir / "mtcnn_clean.sqlite"
shutil.copy2(drive_root / "cache/mtcnn_clean.sqlite", cache_path)

archive_path = data_dir / "img_align_celeba.zip"
print("Copying image archive locally...", flush=True)
shutil.copy2(drive_root / "data/img_align_celeba.zip", archive_path)

partition_path = raw_root / "list_eval_partition.txt"
partition_bytes = partition_path.read_bytes()
rows = [
    line.split()
    for line in partition_bytes.decode("utf-8-sig").splitlines()
]
assert all(len(row) == 2 and row[1] in ("0", "1", "2") for row in rows)

partitions = dict(rows)
assert len(partitions) == len(rows) == 202599
assert Counter(partitions.values()) == {
    "0": 162770, "1": 19867, "2": 19962
}

with sqlite3.connect(
    cache_path.resolve().as_uri() + "?mode=ro", uri=True
) as connection:
    config_row = connection.execute(
        "SELECT value FROM metadata WHERE key = 'config'"
    ).fetchone()
    assert config_row is not None
    cache_config = json.loads(config_row[0])
    assert cache_config["partition_sha256"] == hashlib.sha256(
        partition_bytes
    ).hexdigest()

    records = {
        name: (split, encoded)
        for name, split, encoded in connection.execute(
            "SELECT name, split, result FROM detections"
        )
    }

assert set(records) == set(partitions)

counts = Counter()
with zipfile.ZipFile(archive_path) as archive:
    members = {}
    for member in archive.namelist():
        if member.endswith("/"):
            continue
        if not member.lower().endswith(".jpg"):
            raise ValueError(f"Unexpected archive member: {member}")
        name = Path(member).name
        if name in members:
            raise ValueError(f"Duplicate image filename: {name}")
        members[name] = member

    assert set(members) == set(partitions)

    for name, split in tqdm(rows, desc="Extracting and verifying images"):
        cached_split, encoded = records[name]
        assert cached_split == split, name

        result = json.loads(encoded)
        assert result["status"] in ("ok", "no_face", "invalid_output")
        counts[(split, result["status"])] += 1

        # Reading each ZIP member also checks its CRC.
        content = archive.read(members[name])
        assert hashlib.sha256(content).hexdigest() == (
            result["image_sha256"]
        ), f"Image hash mismatch: {name}"

        (image_dir / name).write_bytes(content)

local_names = {path.name for path in image_dir.iterdir()}
assert local_names == set(partitions)

print("\n=== Cache counts ===")
for (split, status), count in sorted(counts.items()):
    print(f"split={split}, status={status}: {count:,}")

del records

preparation = {
    "source_commit": commit,
    "images_verified": len(local_names),
    "verification": "ZIP CRC and cached SHA-256 for every image",
    "official_split_counts": dict(Counter(partitions.values())),
    "cache_counts": {
        f"{split}/{status}": count
        for (split, status), count in sorted(counts.items())
    },
    "result": "PASS",
    "scope": "Data integrity preparation; no training or test evaluation.",
}
report_path = (
    drive_root / "runs/m3_preparation/day11/full_data_preparation.json"
)
report_path.write_text(
    json.dumps(preparation, indent=2) + "\n", encoding="utf-8"
)

print("Raw data:", raw_root)
print("Cache:", cache_path)
print("Verified images:", len(local_names))
print("Saved:", report_path)
print("M3 full-data preparation: PASS")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
GPU: Tesla T4
PyTorch: 2.11.0+cu130
Source commit: 44e8ac21946d1db1146d1efdbdba36482b3ecfad
Copying image archive locally...


Extracting and verifying images:   0%|          | 0/202599 [00:00<?, ?it/s]


=== Cache counts ===
split=0, status=no_face: 249
split=0, status=ok: 162,521
split=1, status=no_face: 32
split=1, status=ok: 19,835
split=2, status=no_face: 12
split=2, status=ok: 19,950
Raw data: /content/m3_seed0_data/raw
Cache: /content/m3_seed0_data/mtcnn_clean.sqlite
Verified images: 202599
Saved: /content/drive/MyDrive/FacialVisualProfile/runs/m3_preparation/day11/full_data_preparation.json
M3 full-data preparation: PASS


In [2]:
from google.colab import drive

drive.mount("/content/drive", force_remount=True)

Mounted at /content/drive


In [3]:
from pathlib import Path

root = Path("/content/drive/MyDrive/FacialVisualProfile")
print("Project directory accessible:", root.is_dir())

Project directory accessible: True


In [5]:
import hashlib
import importlib
import json
import os
import random
import sys
from importlib import metadata

import numpy as np
import torch
from torch import nn
from torch.utils.data import DataLoader, Subset

os.chdir(workspace)

# Ensure imports come from the formal-training workspace.
for name, module in list(sys.modules.items()):
    if any(
        name == package or name.startswith(package + ".")
        for package in ("datasets", "models", "evaluation")
    ):
        origin = getattr(module, "__file__", None)
        if origin is None or not Path(origin).resolve().is_relative_to(
            workspace.resolve()
        ):
            raise RuntimeError(
                f"Another project module is imported: {name}. "
                "Use a fresh runtime for this notebook."
            )

sys.path.insert(0, str(workspace))
importlib.invalidate_caches()

from datasets.celeba_predicted import CelebAPredictedAttributes
from models.resnet18_geometry import ResNet18GeometryAttributes
from evaluation.attributes import evaluate_attributes

seed = 0
epochs = 10
batch_size = 64
num_workers = 2
device = torch.device("cuda")

def set_seed(value):
    random.seed(value)
    np.random.seed(value)
    torch.manual_seed(value)
    torch.cuda.manual_seed_all(value)

set_seed(seed)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

train_dataset = CelebAPredictedAttributes(
    cache_path, split="train", root=raw_root, failure_policy="mask"
)
valid_dataset = CelebAPredictedAttributes(
    cache_path, split="valid", root=raw_root, failure_policy="mask"
)

assert len(train_dataset) == 162770
assert len(valid_dataset) == 19867
assert train_dataset.attribute_names == valid_dataset.attribute_names

valid_loader = DataLoader(
    valid_dataset,
    batch_size=batch_size,
    shuffle=False,
    num_workers=num_workers,
    pin_memory=True,
    drop_last=False,
)

run_dir = drive_root / "runs/m3_seed0"
source_dir = run_dir / "source"

source_names = [
    "configs/attributes.yaml",
    "datasets/celeba.py",
    "datasets/preprocessing.py",
    "datasets/landmark_cache.py",
    "datasets/celeba_predicted.py",
    "models/resnet18_geometry.py",
    "evaluation/attributes.py",
]
source_hashes = {
    name: hashlib.sha256((workspace / name).read_bytes()).hexdigest()
    for name in source_names
}

config = {
    "model": "ResNet18GeometryAttributes",
    "seed": seed,
    "epochs": epochs,
    "batch_size": batch_size,
    "num_workers": num_workers,
    "optimizer": "Adam",
    "learning_rate": 0.0001,
    "weight_decay": 0.0,
    "precision": "FP32",
    "initial_weights": "ResNet18_Weights.IMAGENET1K_V1",
    "augmentation": "None; use existing shared preprocessing",
    "f1_threshold": 0.5,
    "selection_metric": "validation_mAP",
    "selection_tie_rule": "earliest epoch",
    "attribute_names": train_dataset.attribute_names,
    "epoch_seed_rule": "seed + epoch; epochs start at 1",
    "cudnn_deterministic": True,
    "cudnn_benchmark": False,
    "failure_policy": "retain sample; mask geometry features after MLP",
    "geometry_mlp": [10, 32, 32],
    "geometry_activation": "ReLU after each geometry linear layer",
    "fusion_features": 544,
    "git_commit": commit,
    "source_sha256": source_hashes,
    "environment": {
        "python": sys.version.split()[0],
        "torch": torch.__version__,
        "cuda_runtime": torch.version.cuda,
        "gpu": torch.cuda.get_device_name(0),
        **{
            name: metadata.version(name)
            for name in ("torchvision", "numpy", "Pillow", "scikit-learn")
        },
    },
}

# Confirm shared training settings against the actual baseline config.
baseline_config = json.loads(
    (drive_root / "runs/r18_seed0/config.json")
    .read_text(encoding="utf-8")
)
for key in [
    "seed", "epochs", "batch_size", "num_workers",
    "optimizer", "learning_rate", "weight_decay", "precision",
    "augmentation", "f1_threshold", "selection_metric",
    "attribute_names", "epoch_seed_rule",
    "cudnn_deterministic", "cudnn_benchmark",
]:
    assert config[key] == baseline_config[key], key

config_path = run_dir / "config.json"
if config_path.exists():
    saved_config = json.loads(config_path.read_text(encoding="utf-8"))
    assert saved_config == config, (
        "Existing run configuration differs; do not overwrite it."
    )
else:
    # Protect any existing experiment whose configuration is missing.
    if run_dir.exists() and any(run_dir.iterdir()):
        raise RuntimeError("Run directory is nonempty but config is missing.")
    run_dir.mkdir(parents=True, exist_ok=True)
    config_path.write_text(
        json.dumps(config, indent=2) + "\n", encoding="utf-8"
    )

for name in source_names:
    source = workspace / name
    destination = source_dir / name
    destination.parent.mkdir(parents=True, exist_ok=True)
    if destination.exists():
        assert destination.read_bytes() == source.read_bytes(), name
    else:
        shutil.copy2(source, destination)

checkpoint_paths = sorted(run_dir.glob("epoch_*.pt"))
if checkpoint_paths:
    print("Existing checkpoints found; training cell will restore them.")
    model = ResNet18GeometryAttributes(pretrained=False).to(device)
else:
    # Reset seed immediately before fresh model initialization.
    set_seed(seed)
    model = ResNet18GeometryAttributes(pretrained=True).to(device)

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=config["learning_rate"],
    weight_decay=config["weight_decay"],
)
criterion = nn.BCEWithLogitsLoss()

assert all(p.requires_grad for p in model.parameters())
assert sum(p.numel() for p in model.parameters()) == 11_191_000

if not checkpoint_paths:
    # Include a known failed detection in the batch-64 forward check.
    failure_index = train_dataset.names.index("000199.jpg")
    indices = [i for i in range(63) if i != failure_index]
    assert len(indices) == 63
    indices.append(failure_index)

    smoke_loader = DataLoader(
        Subset(train_dataset, indices),
        batch_size=64,
        num_workers=2,
        shuffle=False,
    )
    images, labels, points, names, valid = next(iter(smoke_loader))
    assert valid[-1].item() is False

    model.eval()
    with torch.inference_mode():
        logits = model(
            images.to(device),
            points.to(device),
            valid.to(device),
        )
        loss = criterion(logits, labels.to(device))

    assert logits.shape == (64, 24)
    assert torch.isfinite(logits).all()
    assert torch.isfinite(loss)
    print("Fresh model batch-64 forward: PASS")
    print("Initial smoke BCE:", round(loss.item(), 6))

    # Epoch 0 contains fresh weights and an unused optimizer.
    checkpoint = {
        "epoch": 0,
        "config": config,
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "history": [],
        "best": None,
    }
    temporary = run_dir / "epoch_00.pt.tmp"
    torch.save(checkpoint, temporary)
    temporary.replace(run_dir / "epoch_00.pt")

    del smoke_loader, images, labels, points, names, valid, logits, loss

print("Train images:", len(train_dataset))
print("Validation images:", len(valid_dataset))
print("Run directory:", run_dir)
print("Saved configuration:", config_path)
print("M3 formal training setup: PASS")

Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 184MB/s]


Fresh model batch-64 forward: PASS
Initial smoke BCE: 0.727568
Train images: 162770
Validation images: 19867
Run directory: /content/drive/MyDrive/FacialVisualProfile/runs/m3_seed0
Saved configuration: /content/drive/MyDrive/FacialVisualProfile/runs/m3_seed0/config.json
M3 formal training setup: PASS


In [ ]:
import json
import math
import time

import numpy as np
import torch
from torch.utils.data import DataLoader
from tqdm.auto import tqdm

# save the history, best as JSON
def save_json(path, value):
    temporary = path.with_name(path.name + ".tmp")
    temporary.write_text(
        json.dumps(value, indent=2, allow_nan=False) + "\n",
        encoding="utf-8",
    )
    temporary.replace(path)


def read_metric(metrics, name):
    matches = [
        value for key, value in metrics.items()
        if key.replace("_", "").lower() == name
    ]
    if len(matches) != 1:
        raise ValueError(f"Unexpected metric keys: {list(metrics)}")
    value = float(matches[0])
    if not math.isfinite(value):
        raise ValueError(f"Non-finite metric: {name}")
    return value


# Check the metric interface before spending time on training.
probe_targets = np.array([[0] * 24, [1] * 24])
probe_logits = np.array([[-10.0] * 24, [10.0] * 24])
probe_metrics = evaluate_attributes(
    probe_logits, probe_targets, threshold=0.5
)
assert read_metric(probe_metrics, "macrof1") == 1.0
assert read_metric(probe_metrics, "map") == 1.0

# Restore the latest completed epoch.
# The following functions are for Restoration 
checkpoint_paths = sorted(run_dir.glob("epoch_*.pt"))
if not checkpoint_paths:
    raise RuntimeError("Run the setup cell to create epoch_00.pt first.")

latest_path = checkpoint_paths[-1]
checkpoint = torch.load(
    latest_path,
    map_location="cpu",
    weights_only=False,  # Load this experiment's own saved checkpoint.
)
assert checkpoint["config"] == config
assert latest_path.stem == f"epoch_{checkpoint['epoch']:02d}"

model.load_state_dict(checkpoint["model_state_dict"], strict=True)
optimizer.load_state_dict(checkpoint["optimizer_state_dict"])

completed_epoch = int(checkpoint["epoch"])
history = checkpoint["history"]
best = checkpoint["best"]

assert [row["epoch"] for row in history] == list(
    range(1, completed_epoch + 1)
)
assert 0 <= completed_epoch <= epochs

del checkpoint
torch.cuda.empty_cache()

# Checkpoint history is authoritative if a previous summary write stopped.
save_json(run_dir / "history.json", history)
if best is not None:
    save_json(run_dir / "best.json", best)

print("Restored:", latest_path.name, flush=True)
print("Completed epochs:", completed_epoch, flush=True)
print("Next epoch:", completed_epoch + 1, flush=True)
print("Training images per epoch:", len(train_dataset), flush=True)
print("Validation images per epoch:", len(valid_dataset), flush=True)


# The real training starts here 
for epoch in range(completed_epoch + 1, epochs + 1):
    # Reset each epoch so epoch-boundary recovery reproduces its sampling.
    set_seed(seed + epoch)

    # build a dataloader for each epoch
    train_loader = DataLoader(
        train_dataset,
        batch_size=batch_size,
        shuffle=True,
        num_workers=num_workers,
        pin_memory=True,
        drop_last=False,
    )

    torch.cuda.reset_peak_memory_stats()
    torch.cuda.synchronize()
    started = time.perf_counter()

    model.train()
    train_loss_sum = 0.0
    train_count = 0
    train_missing = 0

    progress = tqdm(
        train_loader,
        desc=f"Epoch {epoch}/{epochs} train",
    )

    """
    we loop thru each batch in this epoch
    """
    for images, labels, points, names, valid in progress:
        batch_count = images.shape[0]
        train_missing += int((~valid).sum().item())

        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)
        points = points.to(device, non_blocking=True)
        valid = valid.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)
        logits = model(images, points, valid)
        loss = criterion(logits, labels)

        if not torch.isfinite(loss):
            raise RuntimeError(
                f"Non-finite training loss in epoch {epoch}"
            )

        loss.backward()
        optimizer.step()

        train_loss_sum += loss.item() * batch_count
        train_count += batch_count
        progress.set_postfix(
            BCE=f"{train_loss_sum / train_count:.4f}"
        )

    assert train_count == 162770
    assert train_missing == 249

    model.eval()
    valid_loss_sum = 0.0
    valid_count = 0
    valid_missing = 0
    all_logits = []
    all_targets = []

    with torch.inference_mode():
        for images, labels, points, names, valid in tqdm(
            valid_loader,
            desc=f"Epoch {epoch}/{epochs} validation",
        ):
            batch_count = images.shape[0]
            valid_missing += int((~valid).sum().item())
            all_targets.append(labels.numpy())

            images = images.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)
            points = points.to(device, non_blocking=True)
            valid = valid.to(device, non_blocking=True)

            logits = model(images, points, valid)
            loss = criterion(logits, labels)
            if not torch.isfinite(loss) or not torch.isfinite(logits).all():
                raise RuntimeError(
                    f"Non-finite validation output in epoch {epoch}"
                )

            valid_loss_sum += loss.item() * batch_count
            valid_count += batch_count
            all_logits.append(logits.cpu().numpy())

    assert valid_count == 19867
    assert valid_missing == 32

    metrics = evaluate_attributes(
        np.concatenate(all_logits),
        np.concatenate(all_targets),
        threshold=config["f1_threshold"],
    )
    macro_f1 = read_metric(metrics, "macrof1")
    validation_map = read_metric(metrics, "map")

    torch.cuda.synchronize()
    elapsed = time.perf_counter() - started

    row = {
        "epoch": epoch,
        "train_bce": train_loss_sum / train_count,
        "validation_bce": valid_loss_sum / valid_count,
        "validation_macro_f1": macro_f1,
        "validation_mAP": validation_map,
        "train_images": train_count,
        "validation_images": valid_count,
        "train_missing_geometry": train_missing,
        "validation_missing_geometry": valid_missing,
        "elapsed_seconds": elapsed,
        "peak_allocated_gpu_gib": (
            torch.cuda.max_memory_allocated() / 1024**3
        ),
    }
    history.append(row)

    # Strict improvement preserves the earliest epoch on ties.
    if best is None or validation_map > best["validation_mAP"]:
        best = {
            "epoch": epoch,
            "checkpoint": f"epoch_{epoch:02d}.pt",
            "validation_bce": row["validation_bce"],
            "validation_macro_f1": macro_f1,
            "validation_mAP": validation_map,
            "f1_threshold": config["f1_threshold"],
        }

    checkpoint_path = run_dir / f"epoch_{epoch:02d}.pt"
    if checkpoint_path.exists():
        raise RuntimeError(f"Refusing to overwrite {checkpoint_path}")

    temporary = checkpoint_path.with_name(checkpoint_path.name + ".tmp")
    torch.save({
        "epoch": epoch,
        "config": config,
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "history": history,
        "best": best,
    }, temporary)
    temporary.replace(checkpoint_path)

    save_json(run_dir / "history.json", history)
    save_json(run_dir / "best.json", best)

    print(
        f"\nEpoch {epoch}: "
        f"train BCE={row['train_bce']:.6f}, "
        f"val BCE={row['validation_bce']:.6f}, "
        f"F1={macro_f1:.6f}, mAP={validation_map:.6f}, "
        f"time={elapsed / 60:.2f} min",
        flush=True,
    )
    print(
        f"Saved {checkpoint_path.name}; "
        f"best={best['checkpoint']} "
        f"(mAP={best['validation_mAP']:.6f})\n",
        flush=True,
    )

    del train_loader, all_logits, all_targets

print("M3 seed-0 training complete.", flush=True)
print(json.dumps(best, indent=2), flush=True)

Restored: epoch_00.pt
Completed epochs: 0
Next epoch: 1
Training images per epoch: 162770
Validation images per epoch: 19867


Epoch 1/10 train:   0%|          | 0/2544 [00:00<?, ?it/s]

Epoch 1/10 validation:   0%|          | 0/311 [00:00<?, ?it/s]


Epoch 1: train BCE=0.213574, val BCE=0.194043, F1=0.714144, mAP=0.811453, time=9.88 min
Saved epoch_01.pt; best=epoch_01.pt (mAP=0.811453)



Epoch 2/10 train:   0%|          | 0/2544 [00:00<?, ?it/s]

...

Epoch 9/10 validation:   0%|          | 0/311 [00:00<?, ?it/s]


Epoch 9: train BCE=0.053195, val BCE=0.327932, F1=0.711659, mAP=0.773448, time=9.92 min
Saved epoch_09.pt; best=epoch_02.pt (mAP=0.818564)



Epoch 10/10 train:   0%|          | 0/2544 [00:00<?, ?it/s]

Epoch 10/10 validation:   0%|          | 0/311 [00:00<?, ?it/s]


Epoch 10: train BCE=0.041949, val BCE=0.367778, F1=0.715849, mAP=0.771315, time=9.91 min
Saved epoch_10.pt; best=epoch_02.pt (mAP=0.818564)

M3 seed-0 training complete.
{
  "epoch": 2,
  "checkpoint": "epoch_02.pt",
  "validation_bce": 0.1881912535008375,
  "validation_macro_f1": 0.7439260119349894,
  "validation_mAP": 0.8185643813256119,
  "f1_threshold": 0.5
}


In [7]:
import json
from pathlib import Path

import numpy as np
import torch
from torch.utils.data import DataLoader
from tqdm.auto import tqdm

from models.resnet18_geometry import ResNet18GeometryAttributes
from evaluation.attributes import evaluate_attributes

run_path = Path(
    "/content/drive/MyDrive/FacialVisualProfile/runs/m3_seed0"
)

saved_config = json.loads(
    (run_path / "config.json").read_text(encoding="utf-8")
)
saved_best = json.loads(
    (run_path / "best.json").read_text(encoding="utf-8")
)
saved_history = json.loads(
    (run_path / "history.json").read_text(encoding="utf-8")
)

assert [row["epoch"] for row in saved_history] == list(range(1, 11))
assert len(valid_dataset) == 19_867

# Python's max preserves the earliest epoch when scores tie.
selected = max(
    saved_history,
    key=lambda row: row["validation_mAP"],
)
assert selected["epoch"] == saved_best["epoch"]
assert saved_best["checkpoint"] == (
    f"epoch_{saved_best['epoch']:02d}.pt"
)

checkpoint_path = run_path / saved_best["checkpoint"]
checkpoint = torch.load(
    checkpoint_path,
    map_location="cpu",
    weights_only=False,
)
assert checkpoint["epoch"] == saved_best["epoch"]
assert checkpoint["config"] == saved_config

# Load trained weights into a fresh model.
verification_device = torch.device("cuda")
verified_model = ResNet18GeometryAttributes(
    num_attributes=len(saved_config["attribute_names"]),
    pretrained=False,
)
verified_model.load_state_dict(
    checkpoint["model_state_dict"],
    strict=True,
)
verified_model.to(verification_device).eval()
del checkpoint

# Avoid multiprocessing cleanup issues during verification.
verification_loader = DataLoader(
    valid_dataset,
    batch_size=64,
    shuffle=False,
    num_workers=0,
    pin_memory=True,
    drop_last=False,
)

loss_function = torch.nn.BCEWithLogitsLoss()
logit_batches = []
target_batches = []
filenames = []
mask_batches = []
loss_sum = 0.0
sample_count = 0

with torch.inference_mode():
    for images, labels, points, names, valid in tqdm(
        verification_loader,
        desc="Verify best checkpoint",
    ):
        target_batches.append(labels.numpy().copy())
        mask_batches.append(valid.numpy().copy())
        filenames.extend(names)

        images = images.to(verification_device, non_blocking=True)
        labels = labels.to(verification_device, non_blocking=True)
        points = points.to(verification_device, non_blocking=True)
        valid = valid.to(verification_device, non_blocking=True)

        logits = verified_model(images, points, valid)
        loss = loss_function(logits, labels)

        assert torch.isfinite(logits).all()
        assert torch.isfinite(loss)

        batch_size = images.shape[0]
        loss_sum += loss.item() * batch_size
        sample_count += batch_size
        logit_batches.append(logits.cpu().numpy())

logits_array = np.concatenate(logit_batches)
targets_array = np.concatenate(target_batches)
valid_array = np.concatenate(mask_batches)

assert sample_count == 19_867
assert filenames == list(valid_dataset.names)
assert int((~valid_array).sum()) == 32

metrics = evaluate_attributes(
    logits_array,
    targets_array,
    threshold=saved_config["f1_threshold"],
)

def metric_value(name):
    values = [
        value for key, value in metrics.items()
        if key.replace("_", "").lower() == name
    ]
    assert len(values) == 1, f"Unexpected metric keys: {list(metrics)}"
    return float(values[0])

recomputed = {
    "validation_bce": loss_sum / sample_count,
    "validation_macro_f1": metric_value("macrof1"),
    "validation_mAP": metric_value("map"),
}

differences = {
    key: abs(value - float(saved_best[key]))
    for key, value in recomputed.items()
}
tolerance = 1e-6
passed = all(value <= tolerance for value in differences.values())

prediction_path = run_path / "validation_predictions.npz"
with prediction_path.with_suffix(".npz.tmp").open("wb") as file:
    np.savez_compressed(
        file,
        logits=logits_array,
        targets=targets_array,
        filenames=np.asarray(filenames),
        geometry_valid=valid_array,
        attribute_names=np.asarray(saved_config["attribute_names"]),
    )
prediction_path.with_suffix(".npz.tmp").replace(prediction_path)

report = {
    "checkpoint": saved_best["checkpoint"],
    "epoch": saved_best["epoch"],
    "source_commit": saved_config["git_commit"],
    "validation_images": sample_count,
    "missing_geometry": int((~valid_array).sum()),
    "f1_threshold": saved_config["f1_threshold"],
    "recomputed": recomputed,
    "absolute_differences": differences,
    "tolerance": tolerance,
    "passed": passed,
    "num_workers": 0,
    "test_evaluated": False,
}

report_path = run_path / "best_verification.json"
temporary = report_path.with_suffix(".json.tmp")
temporary.write_text(
    json.dumps(report, indent=2, allow_nan=False) + "\n",
    encoding="utf-8",
)
temporary.replace(report_path)

print(json.dumps(report, indent=2))
print("Saved:", prediction_path)
print("Saved:", report_path)

assert passed, "Best-checkpoint metrics differ; inspect the saved report."
print("M3 best-checkpoint verification: PASS")

Verify best checkpoint:   0%|          | 0/311 [00:00<?, ?it/s]

{
  "checkpoint": "epoch_02.pt",
  "epoch": 2,
  "source_commit": "44e8ac21946d1db1146d1efdbdba36482b3ecfad",
  "validation_images": 19867,
  "missing_geometry": 32,
  "f1_threshold": 0.5,
  "recomputed": {
    "validation_bce": 0.1881912535008375,
    "validation_macro_f1": 0.7439260119349894,
    "validation_mAP": 0.8185643813256119
  },
  "absolute_differences": {
    "validation_bce": 0.0,
    "validation_macro_f1": 0.0,
    "validation_mAP": 0.0
  },
  "tolerance": 1e-06,
  "passed": true,
  "num_workers": 0,
  "test_evaluated": false
}
Saved: /content/drive/MyDrive/FacialVisualProfile/runs/m3_seed0/validation_predictions.npz
Saved: /content/drive/MyDrive/FacialVisualProfile/runs/m3_seed0/best_verification.json
M3 best-checkpoint verification: PASS


In [8]:
from pathlib import Path
import json
import zipfile

from google.colab import files

drive_root = Path("/content/drive/MyDrive/FacialVisualProfile")
run_path = drive_root / "runs/m3_seed0"
archive_path = Path("/content/day11_m3_seed0_results.zip")

report = json.loads(
    (run_path / "best_verification.json").read_text(encoding="utf-8")
)
assert report["passed"] is True
assert report["test_evaluated"] is False

sources = {
    "config.json": run_path / "config.json",
    "history.json": run_path / "history.json",
    "best.json": run_path / "best.json",
    "best_verification.json": run_path / "best_verification.json",
    "full_data_preparation.json": (
        drive_root
        / "runs/m3_preparation/day11/full_data_preparation.json"
    ),
}

for path in sources.values():
    assert path.is_file(), f"Missing: {path}"

with zipfile.ZipFile(
    archive_path, "w", compression=zipfile.ZIP_DEFLATED
) as archive:
    for name, path in sources.items():
        archive.write(path, arcname=name)

print("Checkpoints and validation predictions remain on Drive.")
print("Saved:", archive_path)
files.download(str(archive_path))

Checkpoints and validation predictions remain on Drive.
Saved: /content/day11_m3_seed0_results.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>